In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

##Install libraries
| Library | Purpose |
|---|---|
| `transformers` | Load the Hugging Face model |
| `datasets` | Load/manage training data |
| `peft` | LoRA |
| `trl` | SFT training |
| `accelerate` | Training/device management |

In [ ]:
!pip install -U transformers datasets peft trl accelerate torchao

##Import libraries

In [ ]:
import torch

from datasets import Dataset
from transformers import AutoTokenizer

from peft import LoraConfig

from trl import SFTConfig, SFTTrainer

##Create our dataset

In [ ]:
data = [
    {
        "messages": [
            {
                "role": "user",
                "content": "What is a Python variable?"
            },
            {
                "role": "assistant",
                "content": """A variable is a name used to store a value in Python.

Example:

age = 20

Here, age is the variable and 20 is the value stored in it."""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "user",
                "content": "What is a Python function?"
            },
            {
                "role": "assistant",
                "content": """A function is a reusable block of code that performs a specific task.

Example:

def add(a, b):
    return a + b

result = add(10, 20)

The function returns 30."""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "user",
                "content": "What is a Python class?"
            },
            {
                "role": "assistant",
                "content": """A class is a blueprint for creating objects.

Example:

class Student:
    def __init__(self, name):
        self.name = name

student = Student("Rahul")

Here Student is the class and student is an object."""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "user",
                "content": "What is inheritance in Python?"
            },
            {
                "role": "assistant",
                "content": """Inheritance allows one class to acquire properties and methods from another class.

Example:

class Animal:
    def speak(self):
        print("Animal speaks")

class Dog(Animal):
    pass

dog = Dog()
dog.speak()

Output:

Animal speaks"""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "user",
                "content": "What is polymorphism in Python?"
            },
            {
                "role": "assistant",
                "content": """Polymorphism means that the same method name can behave differently for different objects.

Example:

class Dog:
    def speak(self):
        print("Bark")

class Cat:
    def speak(self):
        print("Meow")

dog = Dog()
cat = Cat()

dog.speak()
cat.speak()

Output:

Bark
Meow"""
            }
        ]
    }
]

In [ ]:
#Convert to Hugging Face Dataset
dataset = Dataset.from_list(data)
print(dataset)

In [ ]:
# Inspect the dataset
print(dataset[0])

In [ ]:
# Load the tokenizer
model_name = "Qwen/Qwen3-0.6B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Check:
print(tokenizer)

In [ ]:
# Understand tokenization
# Before training, let's see what happens to text.
text = "What is Python?"
tokens = tokenizer.tokenize(text)
print(tokens)

In [ ]:
# The actual model input is numerical token IDs.
token_ids = tokenizer.encode(text)
print(token_ids)

In [ ]:
# Create the LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

###**r=8**
This is the LoRA rank.
Think: r = capacity of the LoRA adapter
Small:
r = 4
r = 8
More capacity:
r = 16
r = 32
r = 64

Higher r generally means more trainable parameters and potentially more adaptation capacity, at the cost of more memory/compute.

###**lora_alpha=16**
This controls the scaling of the LoRA update.

### **lora_dropout=0.05**
Dropout helps reduce overfitting. 0.05 = 5%

For example:
Training data -> LoRA -> Dropout -> Better generalization

###**target_modules=["q_proj", "v_proj"]**
This tells LoRA: Which layers of the transformer should receive LoRA adapters.
The names:

q_proj

v_proj

refer to parts of the attention mechanism.

Transformer Layer
*   q_proj ← LoRA
*   k_proj
*   v_proj ← LoRA
*   o_proj

In [ ]:
# Create training configuration
training_args = SFTConfig(
    output_dir="./python-tutor-lora",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=1e-4,

    logging_steps=1,

    save_strategy="epoch",

    fp16=True,

    report_to="none",

    max_length=512,

    assistant_only_loss=True
)

Current TRL supports assistant_only_loss=True for conversational datasets; for Qwen3, TRL can patch the chat template appropriately.
Understand the important parameters

### **num_train_epochs=3**
Means:

Dataset -> 1st complete pass -> 2nd complete pass -> 3rd complete pass

One complete pass = one epoch.


### **per_device_train_batch_size=2**
Means the GPU processes approximately:

2 examples -> calculate loss -> update



### **gradient_accumulation_steps=4**
Conceptually:

Batch 1 -> Batch 2 -> Batch 3 -> Batch 4 -> Model update

This lets you simulate a larger effective batch without requiring all examples in GPU memory at once.
Effective batch size here is approximately:
2 × 4 = 8
assuming a single GPU and no distributed training.


### **learning_rate=1e-4**
That's:
0.0001

LoRA commonly uses a higher learning rate than full fine-tuning because only a small set of parameters is being trained; Hugging Face's current PEFT guidance gives around 1e-4 to 2e-4 as typical starting points for LoRA

In [ ]:
# Create the trainer
trainer = SFTTrainer(
    model=model_name,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config
)

In [ ]:
# Start training
trainer.train()

In [ ]:
# See how many parameters are trained After creating the trainer:
trainer.model.print_trainable_parameters()

### **What gets saved? After training:**
python-tutor-lora/

*   adapter_config.json
*   adapter_model.safetensors
*   tokenizer files
*   checkpoints/

The important file is: adapter_model.safetensors

This contains the learned LoRA weights.
It is not a completely new copy of Qwen.

In [ ]:
# Save the adapter. You can explicitly save:
trainer.save_model("./python-tutor-lora")
#and
tokenizer.save_pretrained("./python-tutor-lora")

In [ ]:
#Load the fine-tuned model
# Now let's test our adapter.
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Load base model:
base_model = AutoModelForCausalLM.from_pretrained( model_name,torch_dtype=torch.float16,device_map="auto")

# Then load our LoRA adapter:
model = PeftModel.from_pretrained(base_model,"./python-tutor-lora")

In [ ]:
# Test the model. Let's ask:
messages = [
    {
        "role": "user",
        # "content": "Explain Python inheritance."
        "content": "What is Python variable?"
     }]

In [ ]:
# Create the prompt:
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    # enable_thinking=True
    enable_thinking=False
)


# Tokenize:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)


# Generate:
with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7
    )


# Decode:
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)
# Because our dataset is extremely small, don't expect impressive generalization. The goal here is to understand the pipeline.

In [ ]:
# The complete pipeline At this point you have done:
#                  Hugging Face
#                       │
#                       ▼
#                 Qwen3-0.6B
#                       │
#                       +
#                Python Dataset
#                       │
#                       ▼
#                      SFT
#                       │
#                       +
#                     LoRA
#                       │
#                       ▼
#                 Training
#                       │
#                       ▼
#               LoRA Adapter
#                       │
#                       +
#                 Base Model
#                       │
#                       ▼
#               Fine-tuned LLM

In [ ]:
# What actually changed?
# Before training:
# Qwen
#  ↓
# "What is inheritance?"
#  ↓
# General response

# After training:
# Qwen
#  +
# Python Tutor LoRA
#  ↓
# "What is inheritance?"
#  ↓
# Your desired teaching style

### **LoRA mathematics**
Suppose the original model has a weight matrix:
W

Normal fine-tuning:
W → W'

The whole weight matrix changes.
LoRA instead keeps:
W = frozen

and learns:
A
B

The update is conceptually:
W' = W + BA

where A and B are much smaller matrices.
That's why LoRA is called:
Low-Rank Adaptation

Hugging Face describes LoRA as decomposing the update into smaller low-rank matrices while freezing the pretrained weights.

In [ ]:
# Why LoRA is useful
# Imagine you want three specialized models:
#                  Qwen
#                   │
#        ┌──────────┼──────────┐
#        ↓          ↓          ↓
#    Java LoRA   Python LoRA  AI LoRA
#        │          │          │
#        ↓          ↓          ↓
#  Java Tutor   Python Tutor  AI Tutor

# You don't necessarily need three complete copies of the base model.
# You can maintain:
# Base Model
#     +
# Java Adapter

# Base Model
#     +
# Python Adapter

# Base Model
#     +
# AI Adapter

# Adapters are lightweight compared with full model copies, which is one of PEFT's major advantages.

# **QLoRA**
Base model -> 4-bit quantization -> frozen + LoRA

So:
QLoRA = Quantized model + LoRA

Hugging Face's PEFT/TRL integration supports both LoRA and QLoRA; QLoRA adds 4-bit/8-bit quantization through bitsandbytes.


The four most important code blocks
1. Dataset
  
  dataset = Dataset.from_list(data)


2. LoRA
  
  lora_config = LoraConfig(    
  r=8,    
  lora_alpha=16,    
  lora_dropout=0.05,    
  target_modules=["q_proj", "v_proj"],    
  task_type="CAUSAL_LM"
  )


3. Trainer
  
  trainer = SFTTrainer(    
  model=model_name,    
  args=training_args,    
  train_dataset=dataset,    
  peft_config=lora_config
  )


4. Training
  
  trainer.train()